### ロード

In [11]:
import pandas as pd
import numpy as np
from scipy.stats import skew
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [15]:
# --- ファイル読み込み ---

df_raw = pd.read_parquet("model\macro\liq_engine_raw.parquet", engine="pyarrow")
df_feats_d = pd.read_parquet("model\macro\liq_engine_feats_d.parquet", engine="pyarrow")
df_feats_w = pd.read_parquet("model\macro\liq_engine_feats_w.parquet", engine="pyarrow")

df_feats_d.columns

<>:3: SyntaxWarning: invalid escape sequence '\m'
<>:4: SyntaxWarning: invalid escape sequence '\m'
<>:5: SyntaxWarning: invalid escape sequence '\m'
<>:3: SyntaxWarning: invalid escape sequence '\m'
<>:4: SyntaxWarning: invalid escape sequence '\m'
<>:5: SyntaxWarning: invalid escape sequence '\m'
C:\Users\USER\AppData\Local\Temp\ipykernel_3636\2286936876.py:3: SyntaxWarning: invalid escape sequence '\m'
  df_raw = pd.read_parquet("model\macro\liq_engine_raw.parquet", engine="pyarrow")
C:\Users\USER\AppData\Local\Temp\ipykernel_3636\2286936876.py:4: SyntaxWarning: invalid escape sequence '\m'
  df_feats_d = pd.read_parquet("model\macro\liq_engine_feats_d.parquet", engine="pyarrow")
C:\Users\USER\AppData\Local\Temp\ipykernel_3636\2286936876.py:5: SyntaxWarning: invalid escape sequence '\m'
  df_feats_w = pd.read_parquet("model\macro\liq_engine_feats_w.parquet", engine="pyarrow")


Index(['^GSPC', 'RRPONTSYD', 'DTB3', 'DX-Y.NYB', 'HG=F', 'GC=F', 'DFII10',
       'T10YIE', 'WALCL', 'WDTGAL', 'WCURCIR', 'NFCI', 'STLFSI4', 'ANFCI',
       'NDFACBM027SBOG', 'CPF3M', 'PERMIT', 'RRP_filled', 'Net_Liquidity',
       'Net_Liquidity_z252', 'NFCI_z252', 'Liq_eff', 'CPF3M_DTB3_Spread',
       'CPF3M_DTB3_Spread_z252', 'NDFACBM027SBOG_z252', 'STLFSI4_z252',
       'Cu_Au_Ratio', 'Cu_Au_Ratio_z252', 'PERMIT_GOLD_ratio',
       'PERMIT_GOLD_ratio_z252', 'DXY_z252', 'DFII10_z252', 'ANFCI_z252'],
      dtype='str')

### 特徴量性格診断

In [ ]:
# 特徴量生成
df_feats_w["DFII10_roc4"] = df_feats_w["DFII10"].diff(4)

In [19]:
# リターン生成
lag_weeks = 0
forward_weeks = 4
target_return = f'lag_{lag_weeks}weeks_return_{forward_weeks}weeks'
df_feats_w[target_return] = (
    (df_feats_w["^GSPC"].shift(-(lag_weeks + forward_weeks))) / 
     df_feats_w["^GSPC"].shift(-lag_weeks) -1
)

In [20]:
# stats可視化 median IQR cvar_5
feature = "Liq_eff"
is_inverse = False

df = df_feats_w[[feature, target_return]].copy()
#df = df.loc["2010-01-01":"2016-01-01"]
#df = df.loc["2016-01-01":"2020-01-01"]
#df = df.loc["2020-01-01":"2024-01-01"]
df = df.loc["2024-01-01":"2026-01-01"]
# ランクの作成
df[f'{feature}_rank'] = pd.qcut(df[feature].dropna(), 5, labels=False, duplicates='drop') + 1
if is_inverse:
    df[f'{feature}_rank'] = 6 - df[f'{feature}_rank'] # 1が最悪(重力)、5が最高(浮力)に統一

# クロス集計を作成（行：現在、列：次期）
df['next_rank'] = df[f'{feature}_rank'].shift(-1)
transition_matrix = pd.crosstab(
    df[f'{feature}_rank'], 
    df['next_rank'], 
    normalize='index'
)
# 各統計量の計算
stats = []
for r in range(1, 6):
    d = df[df[f'{feature}_rank'] == r][target_return]
    stats.append({
        'rank': r,
        'median': d.median(),
        'iqr': d.quantile(0.75) - d.quantile(0.25),
        'cvar_5': d[d <= d.quantile(0.05)].mean(),
        'win_rate': (d > 0).mean()
    })
s_df = pd.DataFrame(stats)

fig = make_subplots(
    rows=2, cols=3, 
    subplot_titles=('IQR: Lower is better', 'Median', 'Tail Risk (CVaR 5%)', "Win_Rate", 'Transition Probability (Next Week)'),
    vertical_spacing=0.1,
    specs=[[{"type": "bar"}, {"type": "scatter"}, {"type": "bar"}],
           [{"type": "scatter"}, {"type": "heatmap", "colspan": 2}, None]] # ヒートマップを2列分使う
)

fig.add_trace(go.Bar(x=s_df['rank'], y=s_df['iqr'], name='IQR', marker_color='lightblue'), row=1, col=1)
fig.add_trace(go.Scatter(x=s_df['rank'], y=s_df['median'], name='Median', mode='lines+markers', line_color='orange'), row=1, col=2)
fig.add_trace(go.Bar(x=s_df['rank'], y=s_df['cvar_5'], name='CVaR', marker_color='crimson'), row=1, col=3)
fig.add_trace(go.Scatter(x=s_df['rank'], y=s_df['win_rate'], name='Win Rate', marker_color='crimson'), row=2, col=1)
# 遷移ヒートマップの追加
fig.add_trace(
    go.Heatmap(
        z=transition_matrix.values,
        x=[f"Next R{i}" for i in transition_matrix.columns],
        y=[f"Now R{i}" for i in transition_matrix.index],
        colorscale='Viridis',
        text=np.around(transition_matrix.values, 2), # 数値を表示
        texttemplate="%{text}",
        showscale=False
    ),
    row=2, col=2
)
fig.update_layout(
    title_text=f"Personality Profile: {feature} - Lag:{lag_weeks} weeks, Return:forward {forward_weeks} weeks",
    showlegend=False, template="plotly_dark"
)
fig.show(renderer="browser")


In [ ]:
# 箱ひげ図の描画
plot_df = df.dropna(subset=[target_return, f'{feature}_rank'])

rank_name = f'{feature}_rank'
fig = px.box(
    plot_df,
    x=rank_name,
    y=target_return,
    color=rank_name,
    points="all", # 全データ点（ジッター）を表示して密度を確認
    title=f"Market Physics: S&P 500 {forward_weeks}W Forward Return by {feature}_rank",
    labels={target_return: f'{forward_weeks}-Week Forward Return', '{feature}_rank': 'Liquidity Regime'},
    category_orders={f"{feature}_rank": ['1: Very Low (Heavy)', '2: Low', '3: Neutral', '4: High', '5: Very High (Buoyancy)']}
)

# 0%のライン（損益分岐）を強調
fig.add_hline(y=0, line_dash="dash", line_color="black", annotation_text="Break Even")

# レイアウト調整
fig.update_layout(
    xaxis_title="Liquidity Regime (Liq_eff Z-score)",
    yaxis_title=f"Forward {forward_weeks}W Return (%)",
    showlegend=False,
    template="plotly_white",
    yaxis=dict(tickformat=".1%")
)

# 表示
fig.show(renderer="browser")

### マクロ潮流
2011～2015：  
・実質金利が沈み、クレジットギャップが底を打ちDSRも低下  
・借金返済に追われているため中央銀行がマイナス金利で無重力状態をつくる  
・流動性を入れても経済はまわりにくいが株価は浮く  
2016～2019：  
・期待インフレが安定。実質金利も戻っていく。DSRはゆっくりと上昇  
・インフレなき成長。重量が徐々に正常化。適温  
・ゴールデンレジーム。Liq_effの安定性に注目  
2020～2023：  
・コロナショックで実質金利と期待インフレが底。2022年に向けて歴史的な打ち上げ。DSR急上昇  
・インフレの牙、猛烈な金利引き上げによるショック  
・最も厳しい過渡期扱い。  
2024～2026：  
・実質金利と期待インフレが高い状態で横ばい。DSRがピークアウト  
・ショック後だが重力は戻らない。高い重力市場が適応  
・重量が強いため流動性だけでなく強い体温がないと浮力が相殺される


In [ ]:
df = df_raw[["credit_gap", "dsr", "DFII10", "T10YIE", "WALCL"]].dropna(how="all")
df=df.loc["2007-01-01":]
def zscore(df):
    return (df - df.mean()) / df.std()

credit_gap = df["credit_gap"].dropna().resample("ME").ffill().dropna()
dsr = df["dsr"].dropna().resample("ME").ffill().dropna()
DFII10 = df["DFII10"].dropna().resample("ME").mean().dropna().rolling(6).mean().dropna()
T10YIE = df["T10YIE"].dropna().resample("ME").mean().dropna().rolling(6).mean().dropna()
WALCL = df["WALCL"].dropna().resample("ME").mean().dropna()


In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=credit_gap.index, y=credit_gap, mode='lines', name="credit_gap" ))
fig.add_trace(go.Scatter(x=dsr.index, y=dsr, mode='lines', name="dsr" ))
fig.add_trace(go.Scatter(x=DFII10.index, y=DFII10, mode='lines', name="DFII10" ))
fig.add_trace(go.Scatter(x=T10YIE.index, y=T10YIE, mode='lines', name="T10YIE" ))
fig.add_trace(go.Scatter(x=WALCL.index, y=WALCL, mode='lines', name="WALCL" ))

fig.show(renderer="browser")

pd.set_option("display.max.rows", None)
print(df)


### 

### 